# Golden Path SDG Hub pt-BR (Qualidade Controlada)

Este notebook implementa um pipeline end-to-end com:
- preprocessamento de PDFs em lote com Docling
- chunking automatico (default 1000/200)
- geracao automatica de ICL por PDF (pt-BR)
- seed dataset no schema do flow extractive
- traducao de flow para pt-BR com fallback automatico
- dry run, generate e validacoes de qualidade


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import re
import subprocess
import time

import litellm
import nest_asyncio
from datasets import Dataset
from dotenv import load_dotenv
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

from sdg_hub import Flow, FlowRegistry
from sdg_hub.core.utils.translation import translate_flow

nest_asyncio.apply()


In [ ]:
# Resolve base directory robustly
if (Path.cwd() / '.env').exists():
    BASE_DIR = Path.cwd().resolve()
else:
    BASE_DIR = (Path.cwd() / 'examples' / 'knowledge_tuning' / 'enhanced_summary_knowledge_tuning').resolve()

if not BASE_DIR.exists():
    raise FileNotFoundError(f'Base directory not found: {BASE_DIR}')

ENV_PATH = BASE_DIR / '.env'
if not ENV_PATH.exists():
    raise FileNotFoundError(f'.env not found at: {ENV_PATH}')

load_dotenv(ENV_PATH)

DOC_DIR = BASE_DIR / 'document_collection'
SEED_DATA_PATH = (BASE_DIR / os.getenv('SEED_DATA_PATH', 'sdg_demo_output/seed_data.jsonl')).resolve()
OUTPUT_DIR = (BASE_DIR / os.getenv('OUTPUT_DATA_FOLDER', 'output_data_ptbr')).resolve()
TRANSLATED_FLOWS_DIR = os.getenv('TRANSLATED_FLOWS_DIR', './translated_flows_pt').strip()

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
MAX_CONCURRENCY = int(os.getenv('MAX_CONCURRENCY', '2'))
DRY_RUN_MAX_CONCURRENCY = int(os.getenv('DRY_RUN_MAX_CONCURRENCY', str(MAX_CONCURRENCY)))
FLOW_MAX_CONCURRENCY = int(os.getenv('FLOW_MAX_CONCURRENCY', str(MAX_CONCURRENCY)))
NUMBER_OF_SUMMARIES = int(os.getenv('NUMBER_OF_SUMMARIES', '3'))
FLOW_REQUEST_TIMEOUT_SECONDS = int(os.getenv('FLOW_REQUEST_TIMEOUT_SECONDS', '180'))
FLOW_ASYNC_MODE = os.getenv('FLOW_ASYNC_MODE', 'false').lower() in ('1', 'true', 'yes')
LLM_TIMEOUT_SECONDS = int(os.getenv('LLM_TIMEOUT_SECONDS', '180'))
LLM_RETRIES = int(os.getenv('LLM_RETRIES', '3'))
ICL_MAX_TOKENS = int(os.getenv('ICL_MAX_TOKENS', '400'))
ICL_EXCERPT_MAX_CHARS = int(os.getenv('ICL_EXCERPT_MAX_CHARS', '1200'))
MAX_SEED_ROWS = int(os.getenv('MAX_SEED_ROWS', '80'))
SUMMARY_MAX_TOKENS = int(os.getenv('SUMMARY_MAX_TOKENS', '1024'))
QUESTION_MAX_TOKENS = int(os.getenv('QUESTION_MAX_TOKENS', '192'))
ANSWER_MAX_TOKENS = int(os.getenv('ANSWER_MAX_TOKENS', '1024'))
FAITHFULNESS_MAX_TOKENS = int(os.getenv('FAITHFULNESS_MAX_TOKENS', '256'))

SDG_LANG = os.getenv('SDG_LANG', '').strip()
SDG_LANG_CODE = os.getenv('SDG_LANG_CODE', '').strip()

BASE_FLOW_NAME = 'Extractive Summary Knowledge Tuning Dataset Generation Flow'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f'BASE_DIR: {BASE_DIR}')
print(f'DOC_DIR: {DOC_DIR}')
print(f'SEED_DATA_PATH: {SEED_DATA_PATH}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'SDG_LANG: {SDG_LANG} ({SDG_LANG_CODE})')
print(f'MAX_CONCURRENCY(dry/generate): {DRY_RUN_MAX_CONCURRENCY}/{FLOW_MAX_CONCURRENCY}')
print(f'FLOW_REQUEST_TIMEOUT_SECONDS: {FLOW_REQUEST_TIMEOUT_SECONDS}')
print(f'NUMBER_OF_SUMMARIES: {NUMBER_OF_SUMMARIES}')
print(f'MAX_SEED_ROWS: {MAX_SEED_ROWS}')


In [ ]:
FlowRegistry.discover_flows()

if SDG_LANG and not SDG_LANG_CODE:
    raise ValueError('SDG_LANG is set but SDG_LANG_CODE is missing.')
if SDG_LANG_CODE and not SDG_LANG:
    raise ValueError('SDG_LANG_CODE is set but SDG_LANG is missing.')

if TRANSLATED_FLOWS_DIR:
    translated_path = (BASE_DIR / TRANSLATED_FLOWS_DIR).resolve()
    translated_path.mkdir(parents=True, exist_ok=True)
    FlowRegistry.register_search_path(str(translated_path))
    FlowRegistry._discover_flows(force_refresh=True)

def ensure_translated_flow(base_flow_name: str) -> str:
    if not SDG_LANG:
        return base_flow_name

    translated_name = f"{base_flow_name} ({SDG_LANG})"
    if FlowRegistry.get_flow_path(translated_name) is not None:
        print(f'Found translated flow: {translated_name}')
        return translated_name

    source_path = FlowRegistry.get_flow_path(base_flow_name)
    if source_path is None:
        raise ValueError(f'Base flow not found: {base_flow_name}')

    flow_subdir = Path(source_path).parent.name
    output_dir = None
    if TRANSLATED_FLOWS_DIR:
        output_dir = str((BASE_DIR / TRANSLATED_FLOWS_DIR / f"{flow_subdir}_{SDG_LANG_CODE}").resolve())

    print(f'Translating flow to {SDG_LANG}...')
    translate_flow(
        flow=base_flow_name,
        lang=SDG_LANG,
        lang_code=SDG_LANG_CODE,
        translator_model=os.getenv('TRANSLATOR_MODEL', os.getenv('VLLM_MODEL')),
        verifier_model=os.getenv('VERIFIER_MODEL', os.getenv('TRANSLATOR_MODEL', os.getenv('VLLM_MODEL'))),
        output_dir=output_dir,
        translator_api_key=os.getenv('TRANSLATOR_API_KEY'),
        translator_api_base=os.getenv('TRANSLATOR_API_BASE'),
        verifier_api_key=os.getenv('VERIFIER_API_KEY'),
        verifier_api_base=os.getenv('VERIFIER_API_BASE'),
        register=True,
    )

    return translated_name

FLOW_NAME = ensure_translated_flow(BASE_FLOW_NAME)
FLOW_PATH = FlowRegistry.get_flow_path(FLOW_NAME)
print(f'Flow selected: {FLOW_NAME}')
print(f'Flow path: {FLOW_PATH}')


In [ ]:
if not DOC_DIR.exists():
    raise FileNotFoundError(f'Document directory not found: {DOC_DIR}')

pdf_files = sorted(DOC_DIR.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDF files found in {DOC_DIR}. Add files to document_collection/*.pdf')

print(f'PDF files found: {len(pdf_files)}')

docparser_path = (BASE_DIR.parent / 'docparser_v2.py').resolve()
docling_config_path = (BASE_DIR.parent / 'docling_v2_config.yaml').resolve()

if not docparser_path.exists():
    raise FileNotFoundError(f'docparser_v2.py not found: {docparser_path}')
if not docling_config_path.exists():
    raise FileNotFoundError(f'docling_v2_config.yaml not found: {docling_config_path}')

cmd = [
    'python',
    str(docparser_path),
    '--input-dir', str(DOC_DIR),
    '--output-dir', str(DOC_DIR),
    '--config', str(docling_config_path),
]

print('Running Docling parser...')
subprocess.run(cmd, check=True, cwd=BASE_DIR)

md_files = sorted(DOC_DIR.glob('*.md'))
if not md_files:
    raise FileNotFoundError('No markdown files generated by Docling.')

print(f'Markdown files available: {len(md_files)}')


In [ ]:
splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

pdf_to_chunks = {}

for md_path in md_files:
    text = md_path.read_text(encoding='utf-8')
    text = re.sub(r'-{2,}\|', '-|', text)
    text = re.sub(r'\  +\|', ' |', text)

    docs = splitter.create_documents([text])
    chunks = [d.page_content.strip() for d in docs if d.page_content and d.page_content.strip()]

    if chunks:
        pdf_to_chunks[md_path.stem] = chunks

if not pdf_to_chunks:
    raise RuntimeError('No chunks produced from markdown files.')

total_chunks = sum(len(v) for v in pdf_to_chunks.values())
print(f'Chunking completed. PDFs: {len(pdf_to_chunks)} | Total chunks: {total_chunks}')
for k, v in pdf_to_chunks.items():
    print(f'- {k}: {len(v)} chunks')


In [ ]:
DOMAIN_TEMPLATES = {
    "finance": {
        "domain": "Finance",
        "instruction": "Foque em conformidade, indicadores financeiros, risco e impacto de negocio.",
    },
    "legal": {
        "domain": "Legal",
        "instruction": "Foque em obrigacoes legais, definicoes, escopo normativo e condicoes.",
    },
    "health": {
        "domain": "Health",
        "instruction": "Foque em conceitos clinicos, evidencias, procedimentos e riscos.",
    },
    "technology": {
        "domain": "Technology",
        "instruction": "Foque em arquitetura, operacao, componentes tecnicos e trade-offs.",
    },
    "general": {
        "domain": "General",
        "instruction": "Foque em ideias centrais, fatos relevantes e implicacoes praticas.",
    },
}


def detect_domain(text: str, filename: str) -> str:
    t = f"{filename} {text}".lower()
    if any(k in t for k in ["finance", "bank", "regulation", "fin", "fiscal"]):
        return "finance"
    if any(k in t for k in ["law", "legal", "norma", "decreto", "compliance"]):
        return "legal"
    if any(k in t for k in ["health", "saude", "hospital", "clinico", "medical"]):
        return "health"
    if any(k in t for k in ["software", "api", "model", "llm", "tech", "sistema"]):
        return "technology"
    return "general"


def representative_excerpt(chunks, max_chars=ICL_EXCERPT_MAX_CHARS):
    joined = "\n\n".join(chunks[:3])
    return joined[:max_chars]


def fallback_icl(domain_key: str, excerpt: str) -> dict:
    cfg = DOMAIN_TEMPLATES.get(domain_key, DOMAIN_TEMPLATES["general"])
    short_excerpt = excerpt[:900] if excerpt else "Documento nao disponivel."
    return {
        "icl_document": short_excerpt,
        "icl_query_1": f"Quais sao os pontos principais do documento no dominio {cfg['domain']}?",
        "icl_query_2": f"Quais riscos ou limitacoes relevantes aparecem no documento de {cfg['domain']}?",
        "icl_query_3": f"Qual aplicacao pratica pode ser derivada deste conteudo de {cfg['domain']}?",
    }


def _strip_fences(raw: str) -> str:
    cleaned = raw.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"```$", "", cleaned).strip()
    return cleaned


def _llm_completion_with_retry(messages, *, temperature=0.2, max_tokens=400):
    last_error = None
    for attempt in range(1, LLM_RETRIES + 1):
        try:
            return litellm.completion(
                model=os.getenv("VLLM_MODEL"),
                api_base=os.getenv("VLLM_API_BASE"),
                api_key=os.getenv("VLLM_API_KEY"),
                temperature=temperature,
                max_tokens=max_tokens,
                timeout=LLM_TIMEOUT_SECONDS,
                messages=messages,
            )
        except Exception as exc:
            last_error = exc
            if attempt >= LLM_RETRIES:
                break
            wait_s = min(2 ** attempt, 8)
            print(f"[WARN] LLM attempt {attempt}/{LLM_RETRIES} failed: {exc}. Retrying in {wait_s}s...")
            time.sleep(wait_s)
    raise RuntimeError(f"LLM completion failed after {LLM_RETRIES} attempts: {last_error}")


def generate_icl_with_llm(domain_key: str, excerpt: str) -> dict:
    cfg = DOMAIN_TEMPLATES.get(domain_key, DOMAIN_TEMPLATES["general"])

    system_prompt = (
        "Voce e um especialista em criacao de exemplos de in-context learning para geracao sintetica. "
        "Responda em portugues do Brasil. Retorne APENAS JSON valido."
    )

    user_prompt = f"""
Dominio alvo: {cfg['domain']}
Diretriz de dominio: {cfg['instruction']}

Com base no trecho abaixo, gere um exemplo ICL de alta qualidade.

Requisitos:
1) Texto em portugues do Brasil.
2) JSON com chaves exatas: icl_document, icl_query_1, icl_query_2, icl_query_3
3) Perguntas objetivas, contextualizadas e fieis ao trecho.
4) Nao incluir texto fora do JSON.

Trecho:
{excerpt}
"""

    response = _llm_completion_with_retry(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=ICL_MAX_TOKENS,
    )

    content = (response.choices[0].message.content or "").strip()
    content = _strip_fences(content)
    parsed = json.loads(content)

    required = ["icl_document", "icl_query_1", "icl_query_2", "icl_query_3"]
    for k in required:
        if k not in parsed or not str(parsed[k]).strip():
            raise ValueError(f"Invalid ICL payload. Missing or empty key: {k}")

    return {k: str(parsed[k]).strip() for k in required}


pdf_profiles = {}

for pdf_name, chunks in pdf_to_chunks.items():
    excerpt = representative_excerpt(chunks)
    domain_key = detect_domain(excerpt, pdf_name)

    try:
        icl_fields = generate_icl_with_llm(domain_key, excerpt)
        source_mode = "llm"
    except Exception as exc:
        print(f"[WARN] ICL generation fallback for {pdf_name}: {exc}")
        icl_fields = fallback_icl(domain_key, excerpt)
        source_mode = "fallback"

    pdf_profiles[pdf_name] = {
        "domain": DOMAIN_TEMPLATES.get(domain_key, DOMAIN_TEMPLATES["general"])[
            "domain"
        ],
        "document_outline": pdf_name.replace("_", " ").replace("-", " "),
        "icl_source": source_mode,
        **icl_fields,
    }

print(f"ICL profiles generated: {len(pdf_profiles)}")
for name, profile in pdf_profiles.items():
    print(f"- {name}: domain={profile['domain']} | icl_source={profile['icl_source']}")

In [ ]:
records = []

for pdf_name, chunks in pdf_to_chunks.items():
    profile = pdf_profiles[pdf_name]
    for chunk in chunks:
        records.append({
            'document': chunk,
            'document_outline': profile['document_outline'],
            'domain': profile['domain'],
            'icl_document': profile['icl_document'],
            'icl_query_1': profile['icl_query_1'],
            'icl_query_2': profile['icl_query_2'],
            'icl_query_3': profile['icl_query_3'],
            'source_pdf': pdf_name,
        })

if not records:
    raise RuntimeError('No records generated for seed dataset.')

seed_ds = Dataset.from_list(records)
seed_ds.to_json(str(SEED_DATA_PATH), force_ascii=False)

print(f'Seed dataset saved: {SEED_DATA_PATH}')
print(f'Rows: {len(seed_ds)}')
print(f'Columns: {seed_ds.column_names}')
seed_ds.select(range(min(2, len(seed_ds))))


In [ ]:
required_cols = {
    'document',
    'document_outline',
    'domain',
    'icl_document',
    'icl_query_1',
    'icl_query_2',
    'icl_query_3',
}

missing = required_cols - set(seed_ds.column_names)
assert not missing, f'Missing required columns: {missing}'
print('Schema test passed.')

n_pdfs = len(pdf_to_chunks)
n_unique_icl_documents = len(set(seed_ds['icl_document']))
assert n_unique_icl_documents >= n_pdfs, (
    f'ICL uniqueness test failed: unique_icl={n_unique_icl_documents}, pdfs={n_pdfs}'
)
print('ICL-by-PDF test passed.')

def looks_ptbr(text: str) -> bool:
    t = f" {str(text).lower()} "
    markers = [' de ', ' que ', ' para ', ' com ', ' uma ', ' no ', ' na ', ' e ']
    score = sum(m in t for m in markers)
    return score >= 2

sample_n = min(20, len(seed_ds))
ptbr_ok = 0
for i in range(sample_n):
    if all(looks_ptbr(seed_ds[col][i]) for col in ['icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3']):
        ptbr_ok += 1

assert ptbr_ok >= max(1, sample_n // 2), f'Language test failed: {ptbr_ok}/{sample_n} records look pt-BR.'
print(f'Language test passed: {ptbr_ok}/{sample_n} records look pt-BR.')


In [ ]:
if FLOW_PATH is None:
    raise ValueError(f'Flow path not found for flow: {FLOW_NAME}')

flow = Flow.from_yaml(FLOW_PATH)
flow.set_model_config(
    model=os.getenv('VLLM_MODEL'),
    api_base=os.getenv('VLLM_API_BASE'),
    api_key=os.getenv('VLLM_API_KEY'),
    async_mode=FLOW_ASYNC_MODE,
    timeout=FLOW_REQUEST_TIMEOUT_SECONDS,
)

print(f'Flow loaded: {FLOW_NAME}')
print(f'Default model recommendation: {flow.get_default_model()}')


In [ ]:
run_ds = seed_ds
if MAX_SEED_ROWS > 0 and len(seed_ds) > MAX_SEED_ROWS:
    run_ds = seed_ds.select(range(MAX_SEED_ROWS))
    print(f'[INFO] Limiting run dataset to first {len(run_ds)} rows (MAX_SEED_ROWS={MAX_SEED_ROWS}).')
else:
    print(f'[INFO] Using full seed dataset with {len(run_ds)} rows.')

sample_size = min(2, len(run_ds))
print(f'Running dry run with sample_size={sample_size} and max_concurrency={DRY_RUN_MAX_CONCURRENCY} ...')

dry_result = flow.dry_run(
    run_ds,
    sample_size=sample_size,
    max_concurrency=DRY_RUN_MAX_CONCURRENCY,
    enable_time_estimation=True,
)

print('Dry run completed.')
print(f"Output columns: {dry_result['final_dataset']['columns']}")


In [ ]:
def get_columns(obj):
    if hasattr(obj, 'column_names'):
        return list(obj.column_names)
    if hasattr(obj, 'columns'):
        return list(obj.columns)
    return []

def to_jsonl(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if hasattr(obj, 'to_json'):
        try:
            obj.to_json(str(path), orient='records', lines=True, force_ascii=False)
            return
        except TypeError:
            obj.to_json(str(path), force_ascii=False)
            return
    raise TypeError('Unsupported object type for JSONL export')

runtime_params = {
    'gen_extractive_summary': {
        'n': NUMBER_OF_SUMMARIES,
        'max_tokens': SUMMARY_MAX_TOKENS,
        'temperature': 0.3,
    },
    'question_generation': {
        'max_tokens': QUESTION_MAX_TOKENS,
        'temperature': 0.4,
    },
    'answer_generation': {
        'max_tokens': ANSWER_MAX_TOKENS,
        'temperature': 0.3,
    },
    'eval_faithful_llm_chat': {
        'max_tokens': FAITHFULNESS_MAX_TOKENS,
        'temperature': 0.0,
    },
}

checkpoint_dir = OUTPUT_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print('Running full generation...')
print(f"Running generate with max_concurrency={FLOW_MAX_CONCURRENCY} and NUMBER_OF_SUMMARIES={NUMBER_OF_SUMMARIES}...")
generated_data = flow.generate(
    run_ds,
    runtime_params=runtime_params,
    max_concurrency=FLOW_MAX_CONCURRENCY,
    checkpoint_dir=str(checkpoint_dir),
    save_freq=10,
)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
gen_file = OUTPUT_DIR / f'extractive_summary_gen_{timestamp}.jsonl'
to_jsonl(generated_data, gen_file)

print(f'Generation completed. Output: {gen_file}')
print(f'Columns: {get_columns(generated_data)}')
print(f'Rows: {len(generated_data)}')


In [ ]:
columns = get_columns(generated_data)
judge_col = None
for c in ['faithfulness_judgement', 'faithfulness_judgment']:
    if c in columns:
        judge_col = c
        break

if judge_col is None:
    print('[WARN] Faithfulness column not found in output. Check flow output schema.')
else:
    values = [str(v).strip().upper() for v in generated_data[judge_col]]
    uniq = sorted(set(values))
    print(f'Faithfulness column: {judge_col} | unique values: {uniq}')
    assert set(values) == {'YES'}, f'Faithfulness test failed. Found values: {uniq}'
    print('Faithfulness test passed: output contains only YES.')


## Proximos passos

- Revisar amostras do output para validacao semantica de dominio.
- Ajustar `NUMBER_OF_SUMMARIES` e `MAX_CONCURRENCY` conforme custo/tempo.
- Adicionar templates de dominio extras em `DOMAIN_TEMPLATES` quando necessario.
